In [10]:
# this notebook will scale the data and
# train test split
import pandas as pd
import json
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "mlData"

symbol = "BTCUSDT"
tf = "5m"

src_path = folder_path / "clean" / f"{symbol}-{tf}-vX-cleaned.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
df.head()

,timestamp,label,ROC_3,ROC_5,ROC_10,MOM_3,RETURNS_1,DELTA_1,DELTA_3,VOL_SPIKE,...,RANGE_RATIO,RSI_14,RSI_SLOPE,STOCH_K,CCI_5,DIST_HIGH_5,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS
0,1609199700000,-1,-0.000751,-0.002339,0.005548,-0.189374,-0.002075,-11.871964,-66.426086,0.838335,...,0.930778,60.994850,-3.512627,37.723309,-81.744637,0.838698,0.508031,0.838698,1.485528,0.639149
1,1609200000000,-1,-0.004029,-0.002143,0.001350,-1.014461,-0.001890,-11.295532,-62.670853,1.539922,...,2.090774,56.747784,-9.465221,44.256199,-133.719208,1.444698,1.146976,1.444698,1.146976,0.442562
2,1609200300000,-1,-0.002666,-0.001344,-0.000056,-0.677750,0.001299,-2.558212,-25.725708,0.796544,...,0.804380,58.864796,-7.180096,58.700855,-44.470379,1.074053,1.526613,1.074053,1.526613,0.587009
3,1609200600000,-1,-0.001884,-0.004023,-0.000927,-0.479830,-0.001292,-19.458145,-33.311890,0.865680,...,0.814543,55.929337,-5.065514,44.318054,-79.413910,1.509120,1.201130,1.509120,1.201130,0.443181
4,1609200900000,-1,-0.002710,-0.006660,-0.005351,-0.679541,-0.002715,-25.977829,-47.994186,1.204642,...,1.621373,50.264042,-6.483742,30.484550,-156.113785,1.994144,0.874490,2.136262,0.874490,0.290456


# Train-Test Split prep 
```
After Feature engineering
    │
    ├── Rolling Z (11): DELTA_1, DELTA_3, BUY_RATIO, ATR_5, ATR_14,
                  STOCH_K, DIST_HIGH_5, DIST_HIGH_10, RANGE_POS,
                  DIST_LOW_5, DIST_LOW_10
    │
    ├── Else Global Z  (13): ROC_3, ROC_5, ROC_10, MOM_3, RETURNS_1,
                  ATR_RATIO, ATR_NORM_ROC, RANGE_RATIO,
                  RSI_14, RSI_SLOPE, CCI_5,
                  VOL_SPIKE, DELTA_DIV <-- may have to drop
            
    │
    └── All features
            └── winsorise at ±3σ
            └── drop first 500 rows (rolling warmup)
```

# Must be done in respective order:
- see [Train Data Pipeline](../../../../../journal/trainData-report/trainData-pipeline.md) for full data preparation
- start with df_clean
- Step-by-step
  1. Rolling Z-score (i.e. DELTA_1) — creates NaNs in first 500 rows
  2. Drop first 500 warmup rows — NOW the df is clean
  3. Temporal split into train/val/test — BEFORE any global fitting
  4. Global Z-score — fit on train only, transform val/test
  5. Winsorise — applied after Z-score, on each split separately

In [11]:
# CORRECT at inference — only past 500 bars available
def rolling_zscore_live(series, window=500):
    """
    For each bar t, compute mean and std using only bars t-499 through t.
    This is what rolling().mean() already does — just make it explicit
    in your inference pipeline so you never accidentally use future bars.
    """
    return (series - series.rolling(window).mean()) / \
            series.rolling(window).std()

In [12]:
# STEP:1 500 Rolling adjustment
ROLLING_WINDOW = 500 

ROLLING_FEATURES = [
    # Group L
    "DELTA_1", "DELTA_3", # "BUY_RATIO",
    # Group J
    "ATR_5", "ATR_14",
    # Group K
    "STOCH_K",
    # Group M
    "DIST_HIGH_5", "DIST_HIGH_10", "RANGE_POS", "DIST_LOW_5", "DIST_LOW_10",
]

GLOBAL_FEATURES = [
    # Group I
    "ROC_3", "ROC_5", "ROC_10", "MOM_3", "RETURNS_1",
    # Group J
    "ATR_RATIO", "ATR_NORM_ROC", "RANGE_RATIO",
    # Group K
    "RSI_14", "RSI_SLOPE", "CCI_5",
    # Group M
    "VOL_SPIKE", "DELTA_DIV"
]

# apply rolling Z-score on only rolling features
for col in ROLLING_FEATURES:
    df[col] = rolling_zscore_live(df[col], window=ROLLING_WINDOW)

In [13]:
# df = df.drop(columns=["timestamp"])
print(df.shape)
print(df.columns)
df.head()

(546381, 25)
Index(['timestamp', 'label', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1',
       'DELTA_1', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV', 'ATR_5', 'ATR_14',
       'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_14', 'RSI_SLOPE',
       'STOCH_K', 'CCI_5', 'DIST_HIGH_5', 'DIST_LOW_5', 'DIST_HIGH_10',
       'DIST_LOW_10', 'RANGE_POS'],
      dtype='object')


,timestamp,label,ROC_3,ROC_5,ROC_10,MOM_3,RETURNS_1,DELTA_1,DELTA_3,VOL_SPIKE,...,RANGE_RATIO,RSI_14,RSI_SLOPE,STOCH_K,CCI_5,DIST_HIGH_5,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS
0,1609199700000,-1,-0.000751,-0.002339,0.005548,-0.189374,-0.002075,NaN,NaN,0.838335,...,0.930778,60.994850,-3.512627,NaN,-81.744637,NaN,NaN,NaN,NaN,NaN
1,1609200000000,-1,-0.004029,-0.002143,0.001350,-1.014461,-0.001890,NaN,NaN,1.539922,...,2.090774,56.747784,-9.465221,NaN,-133.719208,NaN,NaN,NaN,NaN,NaN
2,1609200300000,-1,-0.002666,-0.001344,-0.000056,-0.677750,0.001299,NaN,NaN,0.796544,...,0.804380,58.864796,-7.180096,NaN,-44.470379,NaN,NaN,NaN,NaN,NaN
3,1609200600000,-1,-0.001884,-0.004023,-0.000927,-0.479830,-0.001292,NaN,NaN,0.865680,...,0.814543,55.929337,-5.065514,NaN,-79.413910,NaN,NaN,NaN,NaN,NaN
4,1609200900000,-1,-0.002710,-0.006660,-0.005351,-0.679541,-0.002715,NaN,NaN,1.204642,...,1.621373,50.264042,-6.483742,NaN,-156.113785,NaN,NaN,NaN,NaN,NaN


In [14]:
# Trim start-up NaN and must be chonologically ordered
col = df['DELTA_3'] # example column that has NaN warm-up

# Find first and last valid label index
first_valid_idx = col.first_valid_index()
last_valid_idx  = col.last_valid_index()

df = df.loc[first_valid_idx:last_valid_idx].reset_index(drop=True)

print(f"Label NaN trimmed — head up to index {first_valid_idx}, tail after index {last_valid_idx}")
print(f"Rows after trim: {len(df):,}")

# Chronological order must still hold
assert df['timestamp'].is_monotonic_increasing, "Timestamp order broken after trim!"

# Assert no NaN remains anywhere — only contiguous edge NaN in label were expected
remaining_nan = df.isnull().sum()
remaining_nan = remaining_nan[remaining_nan > 0]
assert len(remaining_nan) == 0, f"Unexpected NaN still present:\n{remaining_nan.to_string()}"

print("Chronological order: OK")
print("All columns clean: no NaN remaining.")

Label NaN trimmed — head up to index 499, tail after index 546380
Rows after trim: 545,882
Chronological order: OK
All columns clean: no NaN remaining.


In [15]:
# preview
print(df.shape)
print(df.columns)
df.head()

(545882, 25)
Index(['timestamp', 'label', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1',
       'DELTA_1', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV', 'ATR_5', 'ATR_14',
       'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_14', 'RSI_SLOPE',
       'STOCH_K', 'CCI_5', 'DIST_HIGH_5', 'DIST_LOW_5', 'DIST_HIGH_10',
       'DIST_LOW_10', 'RANGE_POS'],
      dtype='object')


,timestamp,label,ROC_3,ROC_5,ROC_10,MOM_3,RETURNS_1,DELTA_1,DELTA_3,VOL_SPIKE,...,RANGE_RATIO,RSI_14,RSI_SLOPE,STOCH_K,CCI_5,DIST_HIGH_5,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS
0,1609349400000,0,0.004606,0.001757,0.002753,1.282357,0.002613,-0.274840,-0.706248,0.974030,...,0.995959,58.040928,7.895927,1.370281,114.376259,-1.139149,1.071334,-1.024485,0.331995,1.062696
1,1609349700000,0,0.001694,0.005965,0.002874,0.470607,-0.000712,-1.173867,-0.991492,1.515874,...,1.283878,56.445000,2.444511,0.711231,65.921722,-0.516866,0.760792,-0.792436,0.098792,0.728308
2,1609350000000,0,0.002589,0.004583,0.004194,0.725778,0.000689,-0.829945,-1.156085,0.904814,...,0.593160,57.657093,4.074242,0.878751,69.185875,-0.787049,0.448339,-1.011408,0.328199,1.046361
3,1609350300000,0,0.001230,0.003641,0.004205,0.352859,0.001254,-0.578179,-1.308249,0.964004,...,0.805368,59.847393,1.806465,1.480739,111.330124,-1.241895,0.593671,-1.392678,0.696906,1.573174
4,1609350600000,1,0.001812,0.003714,0.000341,0.526269,-0.000132,-0.677621,-1.056272,1.038139,...,0.612290,59.499210,3.054211,0.663838,76.844856,-0.732841,-0.000567,-0.977815,0.665127,1.082636


# Train, test, val split
Temporal split

In [16]:
TRAIN_RATIO     = 0.80
VAL_RATIO       = 0.10
# TEST_RATIO    = 0.10 # implicit — whatever remains


# Sort chronologically first — never shuffle
df = df.sort_values('timestamp').reset_index(drop=True)

def temporal_split(df: pd.DataFrame):
    """
    Strict temporal split — NO shuffling.
    Shuffling causes look-ahead leakage: future bars bleed into train sequences.

    Returns:
        train_df, val_df, test_df
    """
    n        = len(df)
    n_train  = int(n * TRAIN_RATIO)
    n_val    = int(n * VAL_RATIO)

    train_df = df.iloc[:n_train].copy()
    val_df   = df.iloc[n_train : n_train + n_val].copy()
    test_df  = df.iloc[n_train + n_val :].copy()

    print(f"Split sizes — train: {len(train_df):,}  "
          f"val: {len(val_df):,}  "
          f"test: {len(test_df):,}")

    return train_df, val_df, test_df

(train_df,val_df,test_df) = temporal_split(df)
print(f"Trainable — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

Split sizes — train: 436,705  val: 54,588  test: 54,589
Trainable — Train: 436,705 | Val: 54,588 | Test: 54,589


In [17]:

# Verify label distribution is preserved across splits
for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    counts = split["label"].value_counts(normalize=True)
    print(f"{name}: Up={counts[1]:.3f} Down={counts[-1]:.3f} Timeout={counts[0]:.3f}")
    # delete df to preserved memory
del df

Train: Up=0.314 Down=0.326 Timeout=0.360
Val: Up=0.341 Down=0.346 Timeout=0.313
Test: Up=0.332 Down=0.336 Timeout=0.332


In [18]:
# apply scaler
from sklearn.preprocessing import StandardScaler
import joblib
from datetime import date

formatted_date = date.today().strftime("%Y%m")

ALL_FEATURES     = GLOBAL_FEATURES + ROLLING_FEATURES   # final column order

model_name = 'vX'

scaler = StandardScaler()
train_df[GLOBAL_FEATURES] = scaler.fit_transform(train_df[GLOBAL_FEATURES])  # fit + transform on train only
val_df[GLOBAL_FEATURES]   = scaler.transform(val_df[GLOBAL_FEATURES])        # transform only
test_df[GLOBAL_FEATURES]  = scaler.transform(test_df[GLOBAL_FEATURES])       # transform only

joblib.dump(scaler, folder_path / "scaler" / f"{formatted_date}-{model_name}-scaler.pkl")
print("Scaler saved.")


Scaler saved.


# Winsorise 
limited to +- 3SD

In [19]:
WINSOR_CLIP     = 3.0       # clip at ±3σ after Z-score

def winsorise(df: pd.DataFrame, clip: float = WINSOR_CLIP) -> pd.DataFrame:
    """
    Clip all features at ±clip standard deviations.
    Applied AFTER Z-scoring — values are already in σ units so
    
    It replaces the value, it does not remove the row.
    clipping at ±3 means replace > 3sd outlier with 3SD value

    Applied independently per split — no cross-split contamination.
    """
    df = df.copy()
    df[ALL_FEATURES] = df[ALL_FEATURES].clip(lower=-clip, upper=clip)
    return df

print("before: on train_df")
print(train_df.max())
train_df = winsorise(train_df, clip=WINSOR_CLIP)
val_df   = winsorise(val_df,   clip=WINSOR_CLIP)
test_df  = winsorise(test_df,  clip=WINSOR_CLIP)
print("after: on train_df")
print(train_df.max())

before: on train_df
timestamp       1.740434e+12
label           1.000000e+00
ROC_3           4.168481e+01
ROC_5           3.967048e+01
ROC_10          3.069904e+01
MOM_3           1.638533e+01
RETURNS_1       4.515033e+01
DELTA_1         2.070601e+01
DELTA_3         1.799007e+01
VOL_SPIKE       9.055737e+00
DELTA_DIV       1.597355e+00
ATR_5           1.716194e+01
ATR_14          1.383837e+01
ATR_RATIO       1.308251e+01
ATR_NORM_ROC    1.325783e+01
RANGE_RATIO     2.354294e+01
RSI_14          4.355884e+00
RSI_SLOPE       7.400861e+00
STOCH_K         2.082789e+00
CCI_5           1.836442e+00
DIST_HIGH_5     1.140028e+01
DIST_LOW_5      1.084310e+01
DIST_HIGH_10    9.018619e+00
DIST_LOW_10     8.734693e+00
RANGE_POS       2.226820e+00
dtype: float64
after: on train_df
timestamp       1.740434e+12
label           1.000000e+00
ROC_3           3.000000e+00
ROC_5           3.000000e+00
ROC_10          3.000000e+00
MOM_3           3.000000e+00
RETURNS_1       3.000000e+00
DELTA_1         3.

In [20]:
# save train data
# where Y = 0 around 26 cells, remove them during training
train_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-train.jsonl"
val_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-val.jsonl"
test_path  = folder_path / "trainData" / f"{formatted_date}-{model_name}-test.jsonl"

train_path.parent.mkdir(parents=True, exist_ok=True)

train_df.to_json(train_path, orient="records", lines=True)
val_df.to_json(val_path, orient="records", lines=True)
test_df.to_json(test_path,  orient="records", lines=True)

print(f"Saved train: {train_path.name}")
print(f"Saved val: {val_path.name}")
print(f"Saved test:  {test_path.name}")


Saved train: 202603-vX-train.jsonl
Saved val: 202603-vX-val.jsonl
Saved test:  202603-vX-test.jsonl


In [21]:
non_feature = ['timestamp', 'label']
feature_cols = [c for c in train_df.columns if c not in non_feature]

print(f"Train columns  : {len(train_df.columns)}  → {list(train_df.columns)}")

print(f"Scaled columns : {len(GLOBAL_FEATURES)}  → {GLOBAL_FEATURES}")


Train columns  : 25  → ['timestamp', 'label', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1', 'DELTA_1', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV', 'ATR_5', 'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_14', 'RSI_SLOPE', 'STOCH_K', 'CCI_5', 'DIST_HIGH_5', 'DIST_LOW_5', 'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS']
Scaled columns : 13  → ['ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_14', 'RSI_SLOPE', 'CCI_5', 'VOL_SPIKE', 'DELTA_DIV']
